<a href="https://colab.research.google.com/github/Innovatewithapple/RNNProjects/blob/main/RNNBasic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=aa29db801832602ea4ba2a4163a3dcef96acd11f523e0bcb6334e190cfcc6898
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [37]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Input,Bidirectional,LSTM
import tensorflow_datasets as tfds
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from lime.lime_text import LimeTextExplainer

In [38]:
# 1. Download the IMDB dataset
# 'as_supervised=True' gives us (text, label) pairs directly
dataset, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)

# 2. Split into Train and Validation (Standard 80/20 split)
train_data = dataset['train'].batch(32).prefetch(tf.data.AUTOTUNE)
test_data = dataset['test'].batch(32).prefetch(tf.data.AUTOTUNE)

# 3. Get just the sentences for the .adapt() step
# We take a sample of the text to build our vocabulary
train_sentences = dataset['train'].map(lambda x, y: x)

In [39]:
# 1. Grab 3 examples from the 'train' split
for text, label in dataset['train'].take(3):
    print(f"SENTIMENT: {'Positive' if label == 1 else 'Negative'}")
    print(f"REVIEW: {text.numpy().decode('utf-8')[:250]}...") # Show first 250 characters
    print("-" * 50)

SENTIMENT: Negative
REVIEW: This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline...
--------------------------------------------------
SENTIMENT: Negative
REVIEW: I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa...
--------------------------------------------------
SENTIMENT: Negative
REVIEW: Mann photographs the Alberta Rocky Mountains in a superb fashion, and Jimmy Stewart and Walter Brennan give enjoyable performances as they always seem to do. <br /><br />But come on Hollywood - a Mountie telling the people of Dawson City, Yukon to el...
--------------------------------------------------


In [ ]:
# 1. Define the "Resize" for text (e.g., only look at the first 100 words)
max_words = 10000 # Top 10,000 most common words
max_len = 100     # "Resize" every review to 100 words

vectorize_layer = TextVectorization(
    max_tokens=max_words,
    output_mode='int',
    output_sequence_length=max_len
)

# 2. Adapt to your training text (like finding the 'mean' for normalization)
vectorize_layer.adapt(train_sentences)

In [17]:
#SimpleRNN
model_rnn = Sequential([
    Input(shape=(1,), dtype=tf.string), # Input is a raw string
    vectorize_layer,                   # Step 1: Turn string to numbers

    # Step 2: The "Feature Finder" (turns IDs into 64-number vectors)
    Embedding(input_dim=max_words, output_dim=128),

    # Step 3: The "Memory" (The RNN Brick)
    # This is the equivalent of your Conv2D/MaxPool layers!
    SimpleRNN(128),

    # Step 4: The "Head" (Exactly like your CNN!)
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Binary verdict: Positive or Negative
])

In [23]:
#LSTM
# --- STEP 3: THE SKYSCRAPER (Model Build) ---
model_bilstm = Sequential([
    Input(shape=(1,), dtype=tf.string), # One raw string package per example
    vectorize_layer,                   # String -> Integer IDs

    # THE EYES: 128 features per word
    Embedding(input_dim=max_words, output_dim=128),

    # THE BRAIN: Bidirectional LSTM with 64 memory units
    # 'recurrent_dropout' protects the memory between steps
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    #Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),

    # THE JUDGE: Final decision logic
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary answer (0=Neg, 1=Pos)
])

In [18]:
early_stop = EarlyStopping(
    monitor='val_loss',     # 1. Watch the validation loss
    patience=3,              # 2. Wait 3 epochs before giving up
    restore_best_weights=True, # 3. Keep the version that was the 'smartest'
    verbose=1                # 4. Tell us when it stops!
)

In [24]:
model = model_bilstm

In [25]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# It works exactly like your CNN .fit()!
model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 147s 182ms/step - accuracy: 0.6329 - loss: 0.6360 - val_accuracy: 0.6956 - val_loss: 0.5741
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 138s 177ms/step - accuracy: 0.7672 - loss: 0.5054 - val_accuracy: 0.7794 - val_loss: 0.4855
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 132s 169ms/step - accuracy: 0.8436 - loss: 0.3700 - val_accuracy: 0.8093 - val_loss: 0.4440
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 127s 162ms/step - accuracy: 0.8970 - loss: 0.2676 - val_accuracy: 0.8053 - val_loss: 0.4831
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 128s 164ms/step - accuracy: 0.9294 - loss: 0.1967 - val_accuracy: 0.7964 - val_loss: 0.5614
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 127s 162ms/step - accuracy: 0.9554 - loss: 0.1345 - val_accuracy: 0.7905 - val_loss: 0.7054
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


In [31]:
# Create a list of the actual sentences from your validation data
test_texts = []
test_labels = []

# Take the first 100 reviews to have a "pool" to test from
for text, label in test_data.unbatch().take(100):
    test_texts.append(text.numpy().decode('utf-8'))
    test_labels.append(label.numpy())

print(f"✅ Created a list of {len(test_texts)} reviews for LIME to test!")

✅ Created a list of 100 reviews for LIME to test!


In [36]:
# 1. THE BRIDGE: This is the ONLY function you need
def lime_predict(texts):
    # Get the sigmoid prediction (e.g., 0.8)
    preds = model.predict(np.array(texts))
    # LIME needs 2 columns: [Probability of Neg, Probability of Pos]
    # So we turn 0.8 into [0.2, 0.8]
    return np.hstack([1 - preds, preds])

# 2. THE EXPLAINER
from lime.lime_text import LimeTextExplainer
explainer = LimeTextExplainer(class_names=["Negative", "Positive"])

# 3. THE RUN: Pick a review and explain it
idx = 5  # Change this number to see different reviews!
exp = explainer.explain_instance(
    test_texts[idx],
    lime_predict,
    num_features=10
)

# 4. THE SHOW: This creates the heatmap!
exp.show_in_notebook(text=True)

ValueError: Invalid dtype: str25088